# Build a RAG Pipeline — Step by Step

**Goal:** Build a working Retrieval-Augmented Generation system from scratch. You'll chunk documents, create embeddings, store them in a vector database, and query with an LLM — the same architecture behind enterprise AI dashboards.

**Requirements:** OpenAI API key (for embeddings + LLM). A free alternative using local models is provided where possible.

---

## What We're Building

```
Documents → Chunk → Embed → Store in ChromaDB → Query → LLM → Answer
```

By the end, you'll have a system where you can ask natural language questions about a set of documents and get accurate, grounded answers.

In [ ]:
# Uncomment to install
# !pip install chromadb openai sentence-transformers python-dotenv rich tiktoken

import os
import chromadb
import numpy as np
from dotenv import load_dotenv
from sentence_transformers import SentenceTransformer

load_dotenv()

# We'll use local embeddings (free) and optionally OpenAI for generation
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

api_key = os.getenv('OPENAI_API_KEY')
if api_key:
    from openai import OpenAI
    llm_client = OpenAI(api_key=api_key)
    HAS_LLM = True
    print("Embedding model: all-MiniLM-L6-v2 (local, free)")
    print("LLM: OpenAI GPT-4o-mini (API)")
else:
    HAS_LLM = False
    print("Embedding model: all-MiniLM-L6-v2 (local, free)")
    print("LLM: Not available (no API key). RAG retrieval will still work.")

print("\nSetup complete!")

## Step 1: Prepare Your Documents

In production, these would be PDFs, database exports, or API responses. For this exercise, we'll use sample business data that mimics a consulting firm's internal knowledge base.

In [ ]:
# Simulated internal documents from a consulting firm
raw_documents = [
    {
        "title": "Q3 2025 Financial Summary",
        "source": "finance_report_q3",
        "content": """Q3 2025 Financial Performance Summary

Total revenue for Q3 2025 reached $38.4 million, representing a 14% increase 
year-over-year and a 5% decrease from Q2 2025 ($40.5 million). The decline 
from Q2 is attributed primarily to the conclusion of two large contracts 
and seasonal slowdown in the risk and advisory practice.

Revenue by business unit:
- Digital & Cloud: $14.1M (37% of total, +16% YoY)
- Risk & Advisory: $11.2M (29% of total, +6% YoY) 
- Healthcare: $7.8M (20% of total, +12% YoY)
- Energy & Sustainability: $5.3M (14% of total, +4% YoY)

Gross margin improved to 37.2%, up from 35.1% in Q2, driven by higher 
utilization of senior consultants and favorable project mix. Operating 
expenses were held flat at $10.8 million despite headcount growth.

The firm's backlog stands at $98 million, providing approximately 8 months 
of revenue visibility. New bookings in Q3 totaled $32 million, with a 
book-to-bill ratio of 0.88."""
    },
    {
        "title": "Q3 2025 People & Talent Report",
        "source": "hr_report_q3",
        "content": """Q3 2025 People & Talent Report

Total headcount reached 620 employees at the end of Q3, a net increase of 
22 from Q2. The firm hired 32 new consultants during the quarter, with 
10 departures (voluntary attrition rate of 9% annualized, down from 12% 
in Q3 2024).

Key hiring metrics:
- Average time-to-fill: 34 days (target: 30 days)
- Offer acceptance rate: 87%
- Positions filled: 32 of 38 open roles (84% fill rate)
- Internal mobility: 9 promotions, 6 lateral moves

Consultant utilization averaged 78% in Q3, down from 82% in Q2. The 
decline reflects the onboarding ramp for new hires and the seasonal 
slowdown in project starts. Target utilization for Q4 is 80%.

Employee engagement score: 4.1/5.0 (unchanged from Q2). Areas of strength 
include manager effectiveness (4.4) and work-life balance (4.2). Areas for 
improvement include career development clarity (3.6) and compensation 
competitiveness (3.5).

The distributed team grew to 35 members, supporting 10 active 
engagements across Digital & Cloud and Risk & Advisory."""
    },
    {
        "title": "Q3 2025 Client & Business Development Report",
        "source": "client_report_q3",
        "content": """Q3 2025 Client & Business Development Report

The firm onboarded 10 new clients in Q3, bringing the active client 
count to 78. Client retention rate remains strong at 91%.

Notable wins:
- Federal agency: $6.5M multi-year contract for technology modernization
- Mid-size financial institution: $2.8M regulatory compliance engagement
- State agency: $2.1M digital transformation contract
- Enterprise client: $1.5M data analytics and reporting engagement

Average deal size decreased to $320K from $485K in Q2, reflecting a shift 
toward smaller, faster-start engagements. The pipeline contains 38 
qualified opportunities worth $72 million in potential revenue.

Client satisfaction scores averaged 4.5/5.0 across all active engagements, 
the highest quarterly score in firm history. Net Promoter Score improved 
to 58, up from 54 in Q2.

Key account growth: 6 existing clients expanded their engagements in Q3, 
contributing $11.8 million in expansion revenue (31% of total Q3 revenue)."""
    },
    {
        "title": "Q3 2025 Technology & AI Initiatives Report",
        "source": "tech_report_q3",
        "content": """Q3 2025 Technology & AI Initiatives Report

AI Platform Deployment:
The internal AI analytics platform is now used by 68% of senior leadership 
(up from 41% in Q2). The GenAI dashboard processes an average of 280 
queries per day, with a 94% user satisfaction rate. Most common query 
types: revenue breakdown (28%), utilization analysis (22%), client 
pipeline status (19%), and headcount trends (15%).

Cloud Migration:
Azure migration is 85% complete across all business units. Remaining 
workloads (finance and HR legacy systems) are scheduled for Q4. Cloud 
infrastructure costs are tracking 15% below budget due to effective 
resource management and reserved instance utilization.

AI Client Delivery:
Three new AI-focused engagements launched in Q3, including the federal 
technology modernization contract. Total AI-related revenue reached $5.9 million 
(15% of Q3 revenue), up from $2.8 million in Q3 2024 (111% YoY growth).

Security & Compliance:
FedRAMP authorization maintained for all cloud services. Zero security 
incidents in Q3. SOC 2 Type II audit completed with no findings. The 
MCP security framework pilot with three enterprise clients is showing 
promising results, with full deployment planned for Q1 2026."""
    },
]

print(f"Loaded {len(raw_documents)} documents.")
for doc in raw_documents:
    print(f"  - {doc['title']} ({len(doc['content'])} characters)")

## Step 2: Chunking

Documents are too long to use as-is. We split them into smaller chunks that capture individual topics. This ensures retrieval is precise — we get the specific paragraph we need, not the entire 2-page report.

In [ ]:
def chunk_by_paragraph(document, min_chunk_size=100):
    """
    Split document into chunks by paragraph.
    Paragraphs shorter than min_chunk_size are merged with the next one.
    """
    paragraphs = document['content'].split('\n\n')
    chunks = []
    current_chunk = ""
    
    for para in paragraphs:
        para = para.strip()
        if not para:
            continue
        
        if len(current_chunk) + len(para) < min_chunk_size:
            current_chunk += "\n" + para if current_chunk else para
        else:
            if current_chunk:
                chunks.append({
                    'text': current_chunk,
                    'source': document['source'],
                    'title': document['title'],
                })
            current_chunk = para
    
    # Don't forget the last chunk
    if current_chunk:
        chunks.append({
            'text': current_chunk,
            'source': document['source'],
            'title': document['title'],
        })
    
    return chunks


# Chunk all documents
all_chunks = []
for doc in raw_documents:
    chunks = chunk_by_paragraph(doc)
    all_chunks.extend(chunks)
    print(f"{doc['title']}: {len(chunks)} chunks")

print(f"\nTotal chunks: {len(all_chunks)}")
print(f"\nExample chunk:")
print(f"  Source: {all_chunks[2]['source']}")
print(f"  Text: {all_chunks[2]['text'][:200]}...")

## Step 3: Embed and Store in ChromaDB

Now we convert each chunk into a vector and store it in ChromaDB — an open-source, lightweight vector database that's perfect for learning and prototyping.

In [ ]:
# Initialize ChromaDB (in-memory for this exercise)
chroma_client = chromadb.Client()

# Create a collection (like a table in a regular database)
collection = chroma_client.create_collection(
    name="company_knowledge",
    metadata={"description": "Q3 2025 internal reports"}
)

# Embed all chunks and add to ChromaDB
texts = [chunk['text'] for chunk in all_chunks]
embeddings = embedding_model.encode(texts).tolist()
ids = [f"chunk_{i}" for i in range(len(all_chunks))]
metadatas = [{'source': c['source'], 'title': c['title']} for c in all_chunks]

collection.add(
    documents=texts,
    embeddings=embeddings,
    ids=ids,
    metadatas=metadatas,
)

print(f"Stored {collection.count()} chunks in ChromaDB.")
print(f"Each chunk is a {len(embeddings[0])}-dimensional vector.")
print(f"\nThe vector database is ready for semantic search!")

## Step 4: Query the Vector Database

Now the fun part — ask questions in natural language and see what the vector database retrieves.

In [ ]:
def retrieve(query, top_k=3):
    """Retrieve the most relevant chunks for a query."""
    query_embedding = embedding_model.encode([query]).tolist()
    
    results = collection.query(
        query_embeddings=query_embedding,
        n_results=top_k,
    )
    
    print(f"\n🔎 Query: \"{query}\"")
    print(f"\nRetrieved {top_k} most relevant chunks:\n")
    
    for i in range(top_k):
        doc = results['documents'][0][i]
        meta = results['metadatas'][0][i]
        dist = results['distances'][0][i]
        
        print(f"  [{i+1}] Source: {meta['title']}")
        print(f"      Distance: {dist:.4f} (lower = more relevant)")
        print(f"      Text: {doc[:150]}...")
        print()
    
    return results


# Test queries
retrieve("What was our revenue this quarter?")
retrieve("Are employees happy at the firm?")
retrieve("What's happening with AI projects?")

## Step 5: The Full RAG Pipeline — Retrieve + Generate

Now we combine retrieval with LLM generation. The model gets our retrieved context and generates a grounded answer.

In [ ]:
def rag_query(question, top_k=3):
    """Full RAG pipeline: retrieve context → augment prompt → generate answer."""
    
    # Step 1: Retrieve
    query_embedding = embedding_model.encode([question]).tolist()
    results = collection.query(query_embeddings=query_embedding, n_results=top_k)
    
    # Step 2: Build context from retrieved chunks
    context_parts = []
    for i in range(top_k):
        source = results['metadatas'][0][i]['title']
        text = results['documents'][0][i]
        context_parts.append(f"[Source: {source}]\n{text}")
    
    context = "\n\n---\n\n".join(context_parts)
    
    # Step 3: Augmented prompt
    system_prompt = """You are a senior analyst for a consulting firm's C-suite dashboard.
Answer questions based ONLY on the provided context.
If the context doesn't contain the answer, say "I don't have that data."
Always cite which source document supports your answer.
Be concise and quantitative."""
    
    user_prompt = f"""CONTEXT:
{context}

QUESTION: {question}"""
    
    print(f"🔎 Question: \"{question}\"")
    print(f"📄 Retrieved {top_k} relevant chunks from {len(set(r['title'] for r in results['metadatas'][0]))} source documents")
    print(f"{'-'*60}")
    
    # Step 4: Generate (if LLM available)
    if HAS_LLM:
        response = llm_client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt},
            ],
            temperature=0.1,
            max_tokens=500,
        )
        answer = response.choices[0].message.content
        print(f"\n💬 Answer:\n{answer}")
    else:
        print(f"\n📋 Context that would be sent to the LLM:")
        print(f"{context[:500]}...")
        print(f"\n[No LLM available — add OPENAI_API_KEY to .env to see generated answers]")
    
    print(f"\n{'='*60}")
    return context


# Ask questions!
rag_query("How did we perform financially in Q3 compared to Q2?")

In [ ]:
rag_query("What's our employee retention situation?")

In [ ]:
rag_query("How much revenue comes from AI-related work?")

In [ ]:
# Test the guardrails — ask about something not in our data
rag_query("What's the CEO's salary?")

## What You Built

Congratulations — you've built a working RAG pipeline! Here's what each piece does:

| Component | What It Does | Production Equivalent |
|---|---|---|
| `raw_documents` | Source data | Data lake, SharePoint, databases |
| `chunk_by_paragraph()` | Splits docs into searchable pieces | LangChain text splitters, custom pipelines |
| `SentenceTransformer` | Converts text to vectors | OpenAI embeddings, Azure AI Search |
| `ChromaDB` | Stores and searches vectors | Pinecone, Weaviate, Azure AI Search |
| `retrieve()` | Finds relevant context | Retrieval pipeline with reranking |
| `rag_query()` | Full pipeline: retrieve + generate | Enterprise RAG application |
| System prompt | Guardrails and role | Context engineering framework |

### Key Insight: The Model Was Never Trained on This Data

The LLM (GPT-4o-mini) has never seen our Q3 reports. It doesn't "know" our revenue or headcount. RAG retrieved the relevant data and injected it into the prompt — the model generated its answer from that provided context. The model's weights were never modified. That's the fundamental difference between RAG and training.

---

### Next Steps
- **[03-chunking-strategies.ipynb](03-chunking-strategies.ipynb)** — Experiment with different chunking approaches
- **[04-embeddings-explorer.ipynb](04-embeddings-explorer.ipynb)** — Visualize embeddings in 3D
- **[05-rag-vs-long-context.ipynb](05-rag-vs-long-context.ipynb)** — Compare RAG vs. long context approaches